# 第 8 讲练习：BST、层序遍历与树的综合题

先判断题目是否明确给出 BST。只有 BST 才能利用值的大小排除一侧。

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def inorder_values(root):
    result = []
    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        result.append(node.val)
        dfs(node.right)
    dfs(root)
    return result

def sample_bst():
    #        6
    #       / \
    #      2   8
    #     / \ / \
    #    0  4 7  9
    #      / \
    #     3   5
    return TreeNode(6, TreeNode(2, TreeNode(0), TreeNode(4, TreeNode(3), TreeNode(5))), TreeNode(8, TreeNode(7), TreeNode(9)))

## 1. 概念题

1. 为什么只比较节点与直接孩子不足以验证 BST？
2. BST 中序遍历为什么递增？
3. BST 搜索复杂度为什么更准确地写成 `O(h)` 而不是固定 `O(log n)`？
4. 按层 BFS 为什么要保存 `level_size`？
5. 普通二叉树和 BST 的最近公共祖先解法使用的信息有什么不同？

在这里作答。

答：1. 因为BST要求的不仅仅是左右节点和父节点的关系，是左右整个子树之间的关系   
2.BST 中序遍历递增，是因为左子树的所有值都小于当前节点，右子树的所有值都大于当前节点，并且左右子树自身也是 BST。    
3.因为h不一定是log n，只有在树比较平衡时才这样   
4.为了保证每个level之间的输出是分开的   
5.BST只需要比较两个节点的值就可以通过二分法去找到lca，普通二叉树必须递归整个树

## 2. BST 搜索与插入

实现搜索和插入。插入时假设新值不存在，返回根节点。搜索找到时返回对应节点，否则返回 `None`。

In [2]:
def search_bst(root, target):
    if root is None:
        return None
    
    if target == root.val:
        return root
    
    if target < root.val:
         return search_bst(root.left,target)
    else:
        return search_bst(root.right,target)

In [3]:
def insert_into_bst(root, value):
    if root is None:
        return TreeNode(value)
    
    if value < root.val:
        root.left = insert_into_bst(root.left,value)
    else:
        root.right = insert_into_bst(root.right,value)
    return root

In [4]:
root = sample_bst()
assert search_bst(root, 4).val == 4
assert search_bst(root, 10) is None
root = insert_into_bst(root, 1)
assert inorder_values(root) == [0,1,2,3,4,5,6,7,8,9]
root = insert_into_bst(None, 5)
assert root.val == 5 and root.left is None and root.right is None

## 3. 验证 BST

判断整棵树是否满足严格 BST 条件，不允许重复值。不要只比较直接孩子。

任选上下界递归或中序递增方法，并写出函数契约。

In [ ]:
# 上下界递归法

def is_valid_bst(root):
    def valid(root,p,q):# 判断root子树中的所有值是否位于区间(p,q)中的BTS树
        if root is None:
            return True
        if root.val >= q or root.val <= p:
            return False
        
        return valid(root.left,p,root.val) and valid(root.right,root.val,q)
    
    return valid(root,float("-inf"),float("inf"))
        

In [ ]:
# 中序递增法

def is_valid_bst(root):
    previous = float("-inf")
    def inorder(root): #判断当前root树是否是BTS
        nonlocal previous
        if root is None:
            return True
        
        if not inorder(root.left):
            return False
        
        if root.val <=previous:
            return False

        previous = root.val #这个是关键

        if not inorder(root.right):
            return False
        
        

        return True 


    return inorder(root)


In [14]:
assert is_valid_bst(None) is True
assert is_valid_bst(sample_bst()) is True
invalid = TreeNode(5, TreeNode(1), TreeNode(4, TreeNode(3), TreeNode(6)))
assert is_valid_bst(invalid) is False
assert is_valid_bst(TreeNode(2, TreeNode(2), TreeNode(3))) is False

## 4. 第 K 小元素

返回 BST 中第 `k` 小的值。`k` 合法。尝试在找到答案后提前停止，不必先生成完整中序列表。

In [ ]:
# stack迭代写法
def kth_smallest(root, k):
    stack = []
    
    current = root
    while stack or current is not None:
        #先要保证current的所有左节点都进栈
        while current is not None:
            stack.append(current)
            current = current.left
        #目前current是空，栈顶是最左边节点

        current = stack.pop()
        k-=1
        if k==0:
            return current.val

        current = current.right
        


In [ ]:
#函数递归写法

def kth_smallest(root, k):
    def inorder(root):
        nonlocal k

        if root is None:
            return None

        leftresult = inorder(root.left) #先访问左边，看看有没有
        if leftresult is not None: #如果有
            return leftresult
        
        k -=1 
        if k == 0:
            return root.val
        
        rightresult = inorder(root.right)
        
        return rightresult

    return inorder(root)

        

In [20]:
root = sample_bst()
assert kth_smallest(root, 1) == 0
assert kth_smallest(root, 4) == 4
assert kth_smallest(root, 9) == 9

## 5. 层序遍历

按层返回节点值列表。空树返回 `[]`。使用 `deque`。

In [23]:
#BFS，使用queue，一层一层来，先进先出
from collections import deque

def level_order(root):
    if root is None:
        return []

    que = deque()
    que.append(root)

    result_all = []
    while que:
        level = len(que) #先记住当前层有几个元素
        result = []#只记录这一层的元素
        for i in range(level):
            current = que.popleft()
            if current.left is not None:
                que.append(current.left)
            if current.right is not None:
                que.append(current.right)
            result.append(current.val)
        result_all.append(result)
        #这一层结束
    
    return result_all

In [24]:
assert level_order(None) == []
assert level_order(TreeNode(1)) == [[1]]
assert level_order(sample_bst()) == [[6], [2,8], [0,4,7,9], [3,5]]

## 6. 二叉树右视图

返回从右侧能看到的每层最右节点值。可以在层序遍历基础上修改。

本质上还是bfs遍历，最后只输出最外层元素


    

In [26]:
def right_side_view(root):
    if root is None:
        return []

    que = deque()
    que.append(root)

    result_all = []
    while que:
        level = len(que) #先记住当前层有几个元素
        for i in range(level):
            current = que.popleft()
            if current.left is not None:
                que.append(current.left)
            if current.right is not None:
                que.append(current.right)
            
        if i == level-1:
            result_all.append(current.val)
        #这一层结束
    
    return result_all

In [27]:
assert right_side_view(None) == []
assert right_side_view(sample_bst()) == [6,8,9,5]
assert right_side_view(TreeNode(1, TreeNode(2, None, TreeNode(5)), TreeNode(3, None, TreeNode(4)))) == [1,3,4]

## 7. BST 最近公共祖先

`p` 和 `q` 都存在于 BST 中。利用大小关系找到最近公共祖先，优先写迭代版本。

In [ ]:
#要注意的几个点
# 首先是p和q的大小关系，p和q可能是任意顺序的
# 其次是p和q可能本身就是对方的祖先，要判断相等的时候
def lowest_common_ancestor_bst(root, p, q):
    
    current = root
    while current is not None:

        if current.val > q.val and current.val > p.val: #pq一定在左边，去左边找
            current = current.left
        elif current.val < p.val and current.val < q.val: #pq一定在右边，去右边找
            current = current.right
        else: #pq在两边，或者pq本身就是current，当前节点就是LCA
            return current
        

In [33]:
root = sample_bst()
p = search_bst(root, 2)
q = search_bst(root, 8)
assert lowest_common_ancestor_bst(root, p, q).val == 6
p = search_bst(root, 2)
q = search_bst(root, 4)
assert lowest_common_ancestor_bst(root, p, q).val == 2

## 8. 挑战：普通二叉树最近公共祖先

不能利用节点值大小。`p`、`q` 均存在且节点互不相同。比较节点身份，不是只比较值。

下边这个是第一次写出来的，有点问题。is_there函数会搜索一整颗树，最后导致时间复杂度是N^2。是从上往下依次搜索的。

In [ ]:
def lowest_common_ancestor(root, p, q):
    
    def is_there(root,p,q): #搜索p或者q是否在root这棵树中
        if root is None:
            return False
        
        if root is p or root is q:
            return True

        return is_there(root.left, p, q) or is_there(root.right, p, q)
    
    current = root
    while current is not None:
        leftr = is_there(current.left,p,q)
        rightr = is_there(current.right,p,q)
        if leftr and rightr:
            return current
        if leftr:
            current = current.left
        if rightr:
            current = current.right
        if current is p or current is q:
            return current
    

正确的方法应该是自下而上递归搜索

明确lowest_common_ancestor的返回值：如果这棵子树里有p或者q，直接返回这个p或者q的节点。  
如果找到了公共祖先，那就返回祖先的节点   
依次层层上报   

判断是自上而下搜索还是自下而上搜索的方法：
带入一下，看看第一步是发生在底层还是顶层
    

In [45]:
def lowest_common_ancestor(root, p, q):

    if root is None:
        return None
    if root is p or root is q:
        return root
    leftr = lowest_common_ancestor(root.left,p, q)
    rightr = lowest_common_ancestor(root.right,p, q)

    if leftr and rightr:
        return root
    if leftr:
        return leftr
    else:
        return rightr

In [46]:
root = sample_bst()
p = root.left.left       # 0
q = root.left.right.right # 5
assert lowest_common_ancestor(root, p, q) is root.left
assert lowest_common_ancestor(root, root.left, root.right) is root

# 累计复习

## 9. 哈希复习：最长连续序列

要求平均 `O(n)`，不允许排序。注意：为了避免重复起点导致重复扫描，应遍历去重后的集合。

对于x，如果x-1不存在，开始计数

In [ ]:
def longest_consecutive(nums):
    seen = set(nums)
    best = 0
    for x in seen:
        if x-1 not in seen: 
            #正因为seen是set，所以可以O(1)判断x-1是否在seen中。否则这里还得O(n)去遍历一遍nums，时间复杂度就变成O(n^2)了
            #这就是哈希的优势
            length = 1
            k = x
            while k+1 in seen:
                length +=1
                k+=1
            best = max(best,length)
    return best

In [39]:
assert longest_consecutive([]) == 0
assert longest_consecutive([100,4,200,1,3,2]) == 4
assert longest_consecutive([0,0,0,1,2,3]) == 4
assert longest_consecutive([0,3,7,2,5,8,4,6,0,1]) == 9

## 10. 栈复习：每日温度

使用单调栈在 `O(n)` 时间内计算每一天距离下一个更高温度的天数。栈中保存下标。

栈中只存放温度递减的天数，如果遇到高温，就开始弹出

In [43]:
def daily_temperatures(temperatures):
    stack = []
    result = [0]*len(temperatures)
    for i,x in enumerate(temperatures):
        
        if stack:
        
            current = temperatures[stack[-1]]

            while current < x: # 遇到高温
                k = stack.pop()
                result[k] = i-k
                if stack:
                    current = temperatures[stack[-1]]
                else:
                    break
        
        stack.append(i)
    return result

In [44]:
assert daily_temperatures([]) == []
assert daily_temperatures([73,74,75,71,69,72,76,73]) == [1,1,4,2,1,1,0,0]
assert daily_temperatures([90,80,70]) == [0,0,0]

## 11. 总结

1. 哪些题真正利用了 BST 性质？
2. 验证 BST 时，当前节点受到哪些祖先约束？
3. 中序迭代栈中保存的节点表示什么？
4. 层序遍历怎样区分当前层与下一层？
5. 普通树 LCA 的递归返回值表示什么？
6. 两道累计复习题能否不看旧答案完成？

答： 
1. BST搜索和插入利用了BST性质进行二分；验证BST；第k小元素；BST公共祖先   
2.如果当前节点位于某个祖先的右子树，它必须大于该祖先，这形成下界；
如果当前节点位于某个祖先的左子树，它必须小于该祖先，这形成上界；
最终需要同时满足路径上所有祖先留下的上下界。   
3.已经发现了，但是还没有进行处理的节点(这些节点的left子树还没有处理完)   
4.每一层开始前记录当前que的长度   
5.如果这棵子树里有p或者q，直接返回这个p或者q的节点。  
如果找到了公共祖先，那就返回祖先的节点    
6.可以